# Lab 19-11 · Power Straps, Placement, Routing and Clock Trees

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pnr

## Part 1 · Floorplan and a power strap

*Predict* the drop at the middle of Problem 19-11.12's strap (200 µm, 0.5 Ω/µm, both ends fed, 1 mA uniform plus 2 mA at 50 µm).

In [2]:
w, h, rows = pnr.core_dimensions(300_000, 0.60, 1.0)
print(f"Example 1: core {w:.2f} x {h:.2f} um, {rows} rows")
n, L, r = 200, 200.0, 0.5
uniform = pnr.strap_drop_ladder(1e-3, r * L, n)                    # interior nodes 1..199
x = np.arange(1, n) * L / n
point = np.array([2e-3 * r * xx * (L - 50) / L if xx <= 50 else 2e-3 * r * 50 * (L - xx) / L for xx in x])
total = uniform + point
plt.plot(x, 1e3 * uniform, label="uniform 1 mA"); plt.plot(x, 1e3 * total, label="plus 2 mA at 50 um")
plt.xlabel("position (um)"); plt.ylabel("IR drop (mV)"); plt.legend(); plt.show()
print(f"worst {1e3 * total.max():.3f} mV at {x[total.argmax()]:.0f} um; middle {1e3 * total[99]:.3f} mV")
try:
    out = subprocess.run(["ngspice", "-b", "strap.cir"], capture_output=True, text=True, timeout=60).stdout
    print([line for line in out.splitlines() if line.startswith("drop")])
except FileNotFoundError:
    print("ngspice not installed; the README lists the expected values")

Example 1: core 707.11 x 707.20 um, 260 rows
worst 46.875 mV at 50 um; middle 37.500 mV
['drop50 = 0.046875', 'drop100 = 0.0375']


## Part 2 · Quadratic placement

*Predict* where 40 randomly connected cells land when the only fixed pins are 12 pads around a 100 × 100 boundary.

In [3]:
pos = pnr.quadratic_place(4, [("P0", 0, 1), (0, 1, 1), (1, 2, 1), (2, "P1", 1), (1, 3, 1), (3, "P2", 2)],
                         {"P0": (0, 0), "P1": (10, 0), "P2": (5, 6)})
print("Example 3:", np.round(pos, 3).tolist())
rng = np.random.default_rng(7)
pads = {f"P{k}": p for k, p in enumerate([(0, 25), (0, 50), (0, 75), (100, 25), (100, 50), (100, 75),
                                          (25, 0), (50, 0), (75, 0), (25, 100), (50, 100), (75, 100)])}
nets = [(int(a), int(b), 1.0) for a, b in rng.integers(0, 40, size=(80, 2)) if a != b]
nets += [(k, f"P{k % 12}", 1.0) for k in range(0, 40, 3)]
p = pnr.quadratic_place(40, nets, pads)
plt.scatter(p[:, 0], p[:, 1], s=12, label="cells")
plt.scatter(*zip(*pads.values()), marker="s", color="C1", label="pads")
plt.xlim(-5, 105); plt.ylim(-5, 105); plt.gca().set_aspect("equal"); plt.legend(); plt.show()
print("cell spread (std of x, y):", np.round(p.std(axis=0), 1), "against a uniform spread of", round(100 / np.sqrt(12), 1))

Example 3: [[2.5, 1.2], [5.0, 2.4], [7.5, 1.2], [5.0, 4.8]]
cell spread (std of x, y): [6.8 8.3] against a uniform spread of 28.9


## Part 3 · Maze routing and Steiner trees

*Predict* the average and worst RMST/RSMT ratio for random nets of five random pins on a 10 × 10 grid (after duplicates are removed, 18 of the 200 have four distinct pins).

In [4]:
grid = [[(r_ < 6 and c_ == 3) for c_ in range(7)] for r_ in range(7)]
path = pnr.lee_route(grid, (3, 0), (3, 6))
print("Problem 5 path length:", len(path) - 1)
rng = np.random.default_rng(1)
ratios = []
for _ in range(200):
    pts = list(dict.fromkeys(tuple(int(c) for c in q) for q in rng.integers(0, 10, size=(5, 2))))
    if len(pts) < 2:
        continue
    ratios.append(pnr.rmst_length(pts) / pnr.rsmt_length(pts))
plt.hist(ratios, bins=20); plt.xlabel("RMST / RSMT"); plt.ylabel("nets"); plt.show()
print(f"mean {np.mean(ratios):.3f}, max {np.max(ratios):.3f} (Hwang's bound 1.5)")

Problem 5 path length: 12


mean 1.093, max 1.333 (Hwang's bound 1.5)


## Part 4 · A zero-skew merge

*Predict* which way the tap moves as subtree 2's load grows.

In [5]:
r, c = 0.30, 0.20e-15
x, t, C = pnr.merged(10e-12, 40e-15, 12e-12, 60e-15, 500.0, r, c)
print(f"Problem 7: tap {x:.1f} um from subtree 1, delay {t * 1e12:.2f} ps, capacitance {C * 1e15:.0f} fF")
loads = np.linspace(20e-15, 200e-15, 37)
plt.plot(loads * 1e15, [pnr.zero_skew_tap(10e-12, 40e-15, 12e-12, cl, 500.0, r, c) for cl in loads])
plt.xlabel("subtree 2 load (fF)"); plt.ylabel("tap distance from subtree 1 (um)"); plt.show()
try:
    out = subprocess.run(["ngspice", "-b", "merge.cir"], capture_output=True, text=True, timeout=120).stdout
    print([line.split("from")[0].strip() for line in out.splitlines() if line.startswith("elm")])
except FileNotFoundError:
    print("ngspice not installed; the README lists the expected values")

Problem 7: tap 308.3 um from subtree 1, delay 16.55 ps, capacitance 200 fF


['elm1                =   1.65526e-11', 'elm2                =   1.65526e-11']


## Part 5 · Layers

*Predict* the length beyond which metal 3 beats metal 1 for Example 19-11.6's net, with the weak driver and with one eight times stronger.

In [6]:
from scipy.optimize import brentq
layers = {"met1": (1.20565, 0.172375e-15), "met3": (0.166281, 0.214962e-15)}   # open-flow estimates, ohm/um and F/um
CL = 2.302e-15
def delay(L, rd, layer):
    r, c = layers[layer]
    return pnr.wire_delay50(rd, r * L, c * L, CL)   # rd is a Liberty slope R_eff (contains ln 2)
lengths = np.linspace(100, 5000, 50)
for rd, style in ((5640.0, "-"), (705.0, "--")):
    for layer in layers:
        plt.plot(lengths / 1000, [1e12 * delay(L, rd, layer) for L in lengths], style, label=f"{layer}, R_eff = {rd:.0f} ohm")
    cross = brentq(lambda L: delay(L, rd, "met1") - delay(L, rd, "met3"), 1, 1e5)
    print(f"R_eff = {rd:.0f} ohm: metal 3 wins beyond {cross:.0f} um")
plt.xlabel("net length (mm)"); plt.ylabel("50 % delay estimate (ps)"); plt.legend(fontsize=7); plt.show()
try:
    out = subprocess.run(["ngspice", "-b", "net.cir"], capture_output=True, text=True, timeout=120).stdout
    print([line.split("from")[0].split("targ")[0].strip() for line in out.splitlines() if line.startswith(("elmore", "t50"))])
except FileNotFoundError:
    print("ngspice not installed; the README lists the expected values")

R_eff = 5640 ohm: metal 3 wins beyond 3648 um
R_eff = 705 ohm: metal 3 wins beyond 434 um
['elmore              =   1.52793e-09', 't50                 =  1.069991e-09']


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).